In [1]:
import os
import resend
from groq import Groq
from dotenv import load_dotenv
import streamlit as st

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
RESEND_API_KEY = os.getenv("RESEND_API_KEY")
RECEIVER_EMAIL = os.getenv("RECEIVER_EMAIL")

resend.api_key = RESEND_API_KEY
groq_client = Groq(api_key=GROQ_API_KEY)

In [2]:
import json

def send_booking_email(client_name, client_email, booking_details):
    try:
        # Creating a structured dictionary for the booking data
        booking_data = {
            "request_type": "New Booking",
            "client_name": client_name,
            "client_email": client_email,
            "details": booking_details
        }
        
        # Convert dictionary to a clean, indented JSON string
        json_output = json.dumps(booking_data, indent=4, ensure_ascii=False)

        params = {
            "from": "onboarding@resend.dev",
            "to": [RECEIVER_EMAIL],
            "subject": f"New Booking Request: {client_name} (JSON Data)",
            "text": f"New booking received as JSON data:\n\n{json_output}"
        }
        
        response = resend.Emails.send(params)
        return True
    except Exception as e:
        print(f"Error sending email: {e}")
        return False

In [12]:
import csv
import os

# Create a local CSV file for testing/archiving orders if it doesn't exist
CSV_FILE = "restaurant_orders.csv"

if not os.path.exists(CSV_FILE):
    with open(CSV_FILE, mode='w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(["Client Name", "Email", "Order Details", "Order Type"])

def save_order_to_csv(name, email, details, order_type):
    with open(CSV_FILE, mode='a', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow([name, email, details, order_type])

# Updated System Prompt to strictly collect missing information step-by-step
SYSTEM_PROMPT = """
You are a friendly and professional AI assistant for a restaurant. 
Your goal is to help customers place their food orders step by step.

You MUST collect these 4 mandatory pieces of information before finalizing the order:
1. Client Name
2. Client Email
3. Order Details (Items and quantities)
4. Order Type (Delivery or Pickup, with delivery address if delivery)

CRITICAL INSTRUCTIONS:
- Check what information is already provided in the conversation history.
- If ANY of the 4 pieces of information is missing, do NOT output the order block yet. Instead, politely ask the customer ONLY for the missing information.
- ONLY when you have successfully gathered ALL 4 pieces of information across the conversation, explicitly output a summary block formatted strictly like this:
[ORDER_READY]
Name: [Customer Name]
Email: [Customer Email]
Details: [Order Items]
Type: [Delivery/Pickup]
[/ORDER_READY]
"""

In [19]:
def predict_restaurant_order(message, history):
    api_messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    
    for entry in history:
        if isinstance(entry, dict):
            api_messages.append({"role": entry.get("role", "user"), "content": entry.get("content", "")})
        elif isinstance(entry, (list, tuple)) and len(entry) == 2:
            human_msg, assistant_msg = entry
            if human_msg:
                api_messages.append({"role": "user", "content": human_msg})
            if assistant_msg:
                api_messages.append({"role": "assistant", "content": assistant_msg})
                
    api_messages.append({"role": "user", "content": message})
    
    try:
        completion = groq_client.chat.completions.create(
            model="openai/gpt-oss-120b",
            messages=api_messages,
            temperature=0.7,
            max_tokens=500
        )
        reply = completion.choices[0].message.content
    except Exception as e:
        return f"Sorry, I encountered an error: {e}"
    
    if "[ORDER_READY]" in reply and "[/ORDER_READY]" in reply:
        try:
            start_idx = reply.find("[ORDER_READY]") + len("[ORDER_READY]")
            end_idx = reply.find("[/ORDER_READY]")
            order_block = reply[start_idx:end_idx].strip()
            
            order_lines = order_block.split("\n")
            
            if len(order_lines) >= 4:
                c_name = order_lines[0].replace("Name:", "").strip()
                c_email = order_lines[1].replace("Email:", "").strip()
                c_details = order_lines[2].replace("Details:", "").strip()
                c_type = order_lines[3].replace("Type:", "").strip()
                
                if c_name and c_email and c_details and c_type:
                    save_order_to_csv(c_name, c_email, c_details, c_type)
                    email_sent = send_booking_email(c_name, c_email, f"Items: {c_details} | Type: {c_type}")
                    
                    if email_sent:
                        reply += "\n\n✅ *Order successfully placed, saved to archive, and email sent!*"
                    else:
                        reply += "\n\n⚠️ *Order saved locally, but email notification failed.*"
        except Exception as parse_error:
            print(f"Safe parse handling: {parse_error}")
            
    return reply

In [20]:
import gradio as gr

# Launch Gradio Chat Interface linked to our backend function
demo = gr.ChatInterface(
    fn=predict_restaurant_order, 
    title="🍔 Restaurant AI Assistant & Ordering Bot",
    description="Chat with our assistant to place your order. Provide your name, email, order details, and whether it's delivery or pickup."
)

demo.launch(inline=True)

* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.
